# SIEWS+ — Jalan Berlubang & Corrosion Detection Training
## YOLO26n Transfer Learning

**Dataset:** `jalan berlubang.v3-tambahan-dataset.yolo26.zip`
**Kelas terdeteksi:** 3 kelas (class 0: 1633 | class 1: 403 | class 2: 645)

---
**ALUR WAJIB:**
1. Jalankan Cell 1-4 (setup & ekstrak)
2. Cell 5: Distribusi class (angka)
3. **Cell 6: Visualisasi ID angka — perhatikan gambar!**
4. **Cell 7: Isi CLASS_NAMES sesuai gambar**
5. **Cell 8: Konfirmasi ulang dengan nama — pastikan tidak terbalik!**
6. Cell 9+: YAML → Training → Evaluasi

**Kaggle:** Upload ZIP → Add Data → GPU T4 → Run All → Download `.pt`

## 1. Setup

In [ ]:
import os, sys, subprocess, zipfile, shutil, random, yaml, traceback
from pathlib import Path
from collections import Counter

IS_KAGGLE = os.path.exists('/kaggle/input')
print(f'Kaggle: {IS_KAGGLE}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'ultralytics', '-q'], check=True)
import ultralytics, torch
print(f'Ultralytics : {ultralytics.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
DEVICE = '0' if torch.cuda.is_available() else 'cpu'


## 2. Path Config

In [ ]:
ZIP_NAME    = 'jalan berlubang.v3-tambahan-dataset.yolo26.zip'
FOLDER_NAME = 'jalan berlubang.v3-tambahan-dataset.yolo26'  # sudah diekstrak
RUN_NAME    = 'jalan_berlubang'
OUT_NAME    = 'best_stage_jalan_berlubang.pt'

if IS_KAGGLE:
    DATASET_ZIP = None
    OUTPUT_DIR  = Path('/kaggle/working')
    EXTRACT_DIR = None
else:
    DATASET_DIR = Path('F:/migas-siews/dataset')
    folder_path = DATASET_DIR / FOLDER_NAME
    zip_path    = DATASET_DIR / ZIP_NAME
    if folder_path.exists():
        DATASET_ZIP = None
        EXTRACT_DIR = folder_path
        print(f'Folder sudah ada  : {EXTRACT_DIR}')
    elif zip_path.exists():
        DATASET_ZIP = zip_path
        EXTRACT_DIR = Path(f'F:/migas-siews/training/runs_{RUN_NAME}/dataset')
        print(f'ZIP ditemukan     : {DATASET_ZIP}')
    else:
        raise FileNotFoundError(f'Dataset tidak ada!\n  Folder: {folder_path}\n  ZIP   : {zip_path}')

OUTPUT_DIR = Path(f'F:/migas-siews/training/runs_{RUN_NAME}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_DIR / 'runs' / RUN_NAME
print(f'Output dir        : {OUTPUT_DIR}')
print(f'Run dir           : {RUN_DIR}')


## 3. Ekstrak Dataset

In [ ]:
if IS_KAGGLE:
    kaggle_input = Path('/kaggle/input')
    found = None
    for candidate in kaggle_input.rglob('train'):
        if (candidate / 'images').exists():
            found = candidate.parent
            break
    if found is None and (kaggle_input / 'train' / 'images').exists():
        found = kaggle_input
    if found is None:
        subdirs = [d for d in kaggle_input.iterdir() if d.is_dir()]
        print('Isi /kaggle/input/:', [d.name for d in subdirs])
        for sd in subdirs:
            if (sd / 'train' / 'images').exists():
                found = sd; break
    if found is None:
        raise FileNotFoundError('Dataset tidak ditemukan! Pastikan sudah Add Data di Kaggle.')
    EXTRACT_DIR = found
    print(f'Dataset: {EXTRACT_DIR}')
elif DATASET_ZIP and DATASET_ZIP.exists():
    if not EXTRACT_DIR.exists():
        print('Extracting...')
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
        print('Done!')
    else:
        print('Already extracted')
else:
    raise FileNotFoundError(f'ZIP tidak ada: {DATASET_ZIP}')

print()
for split in ['train', 'valid', 'val', 'test']:
    p = EXTRACT_DIR / split / 'images'
    if p.exists():
        ni = len(list(p.glob('*.*')))
        nl_dir = EXTRACT_DIR / split / 'labels'
        nl = len(list(nl_dir.glob('*.txt'))) if nl_dir.exists() else 0
        print(f'[{split:6s}] images={ni:5d}  labels={nl:5d}')

## 4. Baca README

In [ ]:
if DATASET_ZIP and DATASET_ZIP.exists():
    with zipfile.ZipFile(DATASET_ZIP) as zf:
        for fname in ['README.dataset.txt', 'README.roboflow.txt', 'README.txt']:
            if fname in zf.namelist():
                print(f'=== {fname} ===')
                print(zf.read(fname).decode()[:1000])
else:
    for fname in ['README.dataset.txt', 'README.roboflow.txt']:
        rp = EXTRACT_DIR / fname
        if rp.exists():
            print(f'=== {fname} ===')
            print(rp.read_text()[:1000])

## 5. Distribusi Class (Angka — Belum Ada Nama)
**Catat class ID yang ada sebelum visualisasi.**

In [ ]:
counts = Counter()
lbl_dir = EXTRACT_DIR / 'train' / 'labels'
if not lbl_dir.exists():
    lbl_dir = EXTRACT_DIR / 'valid' / 'labels'

for lf in lbl_dir.glob('*.txt'):
    for line in lf.read_text().strip().splitlines():
        if line.strip():
            try: counts[int(line.split()[0])] += 1
            except: pass

if not counts:
    print('TIDAK ADA LABEL! Cek struktur folder.')
else:
    print('=== Distribusi Class (HANYA ANGKA) ===')
    total = sum(counts.values())
    for cid in sorted(counts):
        pct = counts[cid] / total * 100
        bar = '#' * (counts[cid] // 50)
        print(f'  class {cid}: {counts[cid]:5d} instances ({pct:4.1f}%)  {bar}')
    print(f'Jumlah kelas unik : {len(counts)}')
    print(f'Max class ID      : {max(counts)}')
    print()
    print('LANGKAH BERIKUTNYA: Lihat Cell 6 → visualisasi gambar dengan ID angka')

## 6. STEP 1: Visualisasi dengan CLASS ID (Angka)
**Lihat gambar ini. Tiap warna = 1 class ID. Perhatikan objek apa di tiap warna.**

- **ORANGE** = ID 0
- **CYAN** = ID 1
- **RED** = ID 2

**Setelah lihat, baru isi CLASS_NAMES di Cell 7.**

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

COLOR_PALETTE = [
    (255,  80,   0),  # ID 0 orange
    (  0, 200, 255),  # ID 1 cyan
    (255,   0,   0),  # ID 2 red
    (255, 220,   0),  # ID 3 yellow
    (  0, 255, 100),  # ID 4 green
    (180,   0, 255),  # ID 5 purple
]

train_img_dir = EXTRACT_DIR / 'train' / 'images'
train_lbl_dir = EXTRACT_DIR / 'train' / 'labels'
all_imgs = list(train_img_dir.glob('*.jpg')) + list(train_img_dir.glob('*.png'))

# Hanya ambil yang punya label berisi
labeled_imgs = []
for ip in all_imgs:
    lp = train_lbl_dir / (ip.stem + '.txt')
    if lp.exists() and lp.stat().st_size > 0:
        labeled_imgs.append(ip)
    if len(labeled_imgs) >= 100:
        break

if not labeled_imgs:
    raise FileNotFoundError(f'Tidak ada gambar berlabel di {train_img_dir}')

samples = random.sample(labeled_imgs, min(9, len(labeled_imgs)))
fig, axes = plt.subplots(3, 3, figsize=(16, 13))

for ax, ip in zip(axes.flat, samples):
    img = cv2.imread(str(ip))
    if img is None:
        ax.axis('off'); continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lp = train_lbl_dir / (ip.stem + '.txt')
    ids_in_img = []
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5: continue
            cid = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = int((xc-bw/2)*w); y1 = int((yc-bh/2)*h)
            x2 = int((xc+bw/2)*w); y2 = int((yc+bh/2)*h)
            col = COLOR_PALETTE[cid % len(COLOR_PALETTE)]
            cv2.rectangle(img, (x1,y1), (x2,y2), col, 3)
            lbl = f'ID:{cid}'
            (tw,th),_ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
            cv2.rectangle(img, (x1, max(y1-th-8,0)), (x1+tw+6, max(y1,th+8)), (0,0,0), -1)
            cv2.putText(img, lbl, (x1+3, max(y1-4,th+4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, col, 2)
            ids_in_img.append(cid)
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'IDs: {sorted(set(ids_in_img))}', fontsize=11, fontweight='bold')

legend = ' | '.join([f'ID {i}={["ORANGE","CYAN","RED","YELLOW","GREEN"][i]}' for i in sorted(counts.keys())])
plt.suptitle(f'STEP 1 — CLASS ID (ANGKA SAJA)\n{legend}\nPerhatikan tiap warna = objek apa?',
             fontsize=12, fontweight='bold', color='darkred')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR/'step1_class_ids.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Selesai Step 1. Sekarang isi CLASS_NAMES di Cell 7.')

## 7. Isi CLASS_NAMES Berdasarkan Visualisasi

**Setelah lihat Cell 6, konfirmasi nama class di bawah ini.**

Dataset `jalan berlubang v3` kemungkinan besar berisi:
- `0: lubang` — pothole / jalan berlubang (instances terbanyak)
- `1: retak` — crack / retakan aspal
- `2: tambalan` — patched / bekas tambal lubang

**Jika visualisasi Cell 6 menunjukkan berbeda → koreksi di Cell 14 sebelum lanjut!**


In [ ]:
# Nama class berdasarkan analisis dataset jalan berlubang v3
# Verifikasi dengan visualisasi Cell 6 — jika berbeda, koreksi di sini lalu re-run Cell 15
CLASS_NAMES = {
    0: 'lubang',     # pothole — class terbanyak (2910 instances)
    1: 'retak',      # crack   — paling sedikit (904 instances)
    2: 'tambalan',   # patched — bekas tambal (1258 instances)
}
NC = len(CLASS_NAMES)

max_id = max(counts) if counts else -1
print(f'NC             : {NC}')
print(f'Max class ID   : {max_id}')
if max_id >= NC:
    print(f'PERINGATAN: Ada class ID {max_id} tapi NC hanya {NC}!')
else:
    print('OK — jumlah class sesuai')
print()
print('Class names yang akan dipakai:')
for cid, nm in CLASS_NAMES.items():
    print(f'  {cid}: {nm}  ({counts.get(cid, 0)} instances)')
print()
print('Jika nama tidak sesuai gambar Cell 6, koreksi CLASS_NAMES di atas!')


## 8. STEP 2: Konfirmasi Visualisasi dengan Nama
**Pastikan label nama sudah benar dan tidak terbalik sebelum training!**

In [ ]:
for name in CLASS_NAMES.values():
    if '[GANTI' in str(name):
        raise ValueError('CLASS_NAMES masih placeholder! Isi dulu di Cell 7.')

samples2 = random.sample(labeled_imgs, min(9, len(labeled_imgs)))
fig, axes = plt.subplots(3, 3, figsize=(16, 13))

for ax, ip in zip(axes.flat, samples2):
    img = cv2.imread(str(ip))
    if img is None:
        ax.axis('off'); continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lp = train_lbl_dir / (ip.stem + '.txt')
    names_found = []
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5: continue
            cid = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = int((xc-bw/2)*w); y1 = int((yc-bh/2)*h)
            x2 = int((xc+bw/2)*w); y2 = int((yc+bh/2)*h)
            col = COLOR_PALETTE[cid % len(COLOR_PALETTE)]
            nm = CLASS_NAMES.get(cid, f'cls_{cid}')
            cv2.rectangle(img, (x1,y1), (x2,y2), col, 2)
            (tw,th),_ = cv2.getTextSize(nm, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
            cv2.rectangle(img, (x1, max(y1-th-8,0)), (x1+tw+6, max(y1,th+8)), (0,0,0), -1)
            cv2.putText(img, nm, (x1+3, max(y1-4,th+4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, col, 2)
            names_found.append(nm)
    ax.imshow(img); ax.axis('off')
    ax.set_title(', '.join(set(names_found)) or '-', fontsize=10)

plt.suptitle('STEP 2 — KONFIRMASI NAMA CLASS\nApakah label sudah benar? Jika ya, lanjut ke Cell 9.',
             fontsize=12, fontweight='bold', color='darkblue')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR/'step2_confirmed_names.png'), dpi=100, bbox_inches='tight')
plt.show()

# Distribusi dengan nama
print('Distribusi dengan nama:')
for cid in sorted(counts):
    print(f'  {CLASS_NAMES.get(cid, f"cls_{cid}")}: {counts[cid]} instances')
print('\nJika nama SALAH, kembali ke Cell 7 dan perbaiki sebelum training!')

## 9. Buat data.yaml

In [ ]:
val_split  = 'valid' if (EXTRACT_DIR/'valid'/'images').exists() else 'val'
test_split = 'test'  if (EXTRACT_DIR/'test'/'images').exists()  else val_split

DATA_YAML = OUTPUT_DIR / f'{RUN_NAME}.yaml'
yaml.dump({
    'path' : str(EXTRACT_DIR),
    'train': 'train/images',
    'val'  : f'{val_split}/images',
    'test' : f'{test_split}/images',
    'nc'   : NC,
    'names': CLASS_NAMES,
}, open(DATA_YAML, 'w'), default_flow_style=False)
print(f'YAML: {DATA_YAML}')
print(DATA_YAML.read_text())

## 10. Training

In [ ]:
from ultralytics import YOLO

EPOCHS   = 100
BATCH    = 16
IMGSZ    = 640
PATIENCE = 0    # 0 = disable EarlyStopping
FREEZE   = 10

print(f'epochs={EPOCHS}, batch={BATCH}, imgsz={IMGSZ}, device={DEVICE}')
print(f'EarlyStopping: {"DISABLED" if PATIENCE==0 else f"patience={PATIENCE}"}')

TRAIN_OK = False
try:
    model = YOLO('yolo26n.pt')
    results = model.train(
        data=str(DATA_YAML), epochs=EPOCHS, batch=BATCH, imgsz=IMGSZ,
        device=DEVICE, patience=PATIENCE, freeze=FREEZE, pretrained=True,
        project=str(OUTPUT_DIR/'runs'), name=RUN_NAME, exist_ok=True,
        augment=True, mosaic=1.0, mixup=0.1, copy_paste=0.1,
        hsv_h=0.02, hsv_s=0.75, hsv_v=0.4,
        fliplr=0.5, degrees=8.0, scale=0.5, erasing=0.3,
        optimizer='AdamW', lr0=0.001, lrf=0.01, weight_decay=0.0005, warmup_epochs=3,
        save_period=10, verbose=True, plots=True,
    )
    print('Training selesai!')
    TRAIN_OK = True
except KeyboardInterrupt:
    print('Dihentikan manual. Checkpoint:', RUN_DIR/'weights'/'last.pt')
except Exception as e:
    print(f'Error: {e}'); traceback.print_exc()

## 11. Evaluasi

In [ ]:
best_pt = RUN_DIR/'weights'/'best.pt'
last_pt = RUN_DIR/'weights'/'last.pt'
eval_pt = best_pt if best_pt.exists() else last_pt

if not eval_pt.exists():
    print('Tidak ada checkpoint!')
else:
    try:
        if TRAIN_OK:
            m = results.results_dict
            map50=m.get('metrics/mAP50(B)',0); prec=m.get('metrics/precision(B)',0); rec=m.get('metrics/recall(B)',0)
        else:
            em=YOLO(str(eval_pt)); vr=em.val(data=str(DATA_YAML),device=DEVICE,verbose=False)
            map50=vr.box.map50; prec=vr.box.mp; rec=vr.box.mr
        print(f'mAP50     : {map50:.4f}  {"OK" if map50>=0.70 else "Perlu improvement"}')
        print(f'Precision : {prec:.4f}')
        print(f'Recall    : {rec:.4f}')
    except Exception as e:
        print(f'Eval error: {e}')

from IPython.display import Image as IPImg, display
for pn in ['results.png','confusion_matrix.png','PR_curve.png']:
    pf = RUN_DIR/pn
    if pf.exists(): print(f'--- {pn} ---'); display(IPImg(str(pf), width=750))

## 12. Inference Test

In [ ]:
if eval_pt.exists():
    bm = YOLO(str(eval_pt))
    tdir = EXTRACT_DIR/'test'/'images'
    if not tdir.exists(): tdir = EXTRACT_DIR/val_split/'images'
    imgs = list(tdir.glob('*.jpg')) + list(tdir.glob('*.png'))
    if not imgs:
        print('Tidak ada gambar test')
    else:
        samples3 = random.sample(imgs, min(9, len(imgs)))
        fig, axes = plt.subplots(3, 3, figsize=(16, 13))
        for ax, ip in zip(axes.flat, samples3):
            try:
                pred = bm.predict(str(ip), conf=0.25, verbose=False)[0]
                ax.imshow(cv2.cvtColor(pred.plot(), cv2.COLOR_BGR2RGB))
                bxs = pred.boxes
                title = ', '.join([f"{pred.names[int(b.cls[0])]}:{float(b.conf[0]):.0%}" for b in bxs]) if bxs and len(bxs) else 'no det'
                ax.set_title(title[:45], fontsize=8)
            except Exception as e:
                ax.set_title(str(e)[:30])
            ax.axis('off')
        plt.suptitle('Inference Test', fontsize=13)
        plt.tight_layout()
        plt.savefig(str(OUTPUT_DIR/'inference.png'), dpi=100)
        plt.show()

## 13. Auto-Save Model

dest = Path('/kaggle/working') if IS_KAGGLE else Path('F:/migas-siews/backend/models/New')
dest.mkdir(parents=True, exist_ok=True)

best_dst = dest / OUT_NAME
last_dst = dest / OUT_NAME.replace('best_', 'last_')

for src, dst in [(best_pt, best_dst), (last_pt, last_dst)]:
    if src.exists():
        shutil.copy2(src, dst)
        print(f'Saved: {dst}  ({dst.stat().st_size/1e6:.1f} MB)')
    else:
        print(f'Tidak ada: {src}')

if IS_KAGGLE:
    print(f'\nDownload {OUT_NAME} dari tab Output Kaggle')
    print(f'Salin ke: F:/migas-siews/backend/models/New/{OUT_NAME}')
else:
    print(f'\nModel tersimpan di: {best_dst}')


## Troubleshooting

| Masalah | Solusi |
|---|---|
| `IS_KAGGLE not defined` | Jalankan Cell 1 (Setup) dulu sebelum Cell lain |
| Folder tidak ditemukan | Cek `FOLDER_NAME` di Cell 4 sesuai nama folder di `dataset/` |
| Class names terbalik | Koreksi Cell 14, re-run Cell 15 dan 16 |
| mAP rendah | Kurangi `freeze=5`, tambah `epochs=150` |
| OOM | Kurangi `batch=8` atau `imgsz=416` |
| Loss tidak turun | Kurangi `lr0=0.0005` |
